# Block and Forward Pass

How the baseline model processes tokens into predictions. The whole pipeline is: token IDs → embeddings → 9 rounds of attention + MLP → vocabulary predictions → loss.

## The Pipeline

Take the sentence "The cat sat on the ___". The model predicts what comes next at every position.

**Embedding.** Each token ID becomes a 512-dimensional vector. "The" → 512 numbers, "cat" → different 512 numbers. At this point every occurrence of "the" is identical. No context yet. This is `x`. We save a frozen copy called `x0`.

**9 blocks.** Each block does two things in order:
- **Attention** — every position looks at earlier positions and asks "what's relevant to me?" The word "sat" might pull information from "cat" (who sat?) and update its representation to encode that relationship.
- **MLP** — each position independently processes its 512 numbers. Digesting what attention gathered.

After one block, each token's 512 numbers are a little smarter. After 9 blocks, much smarter. The shape never changes — always (batch, seq, 512). What changes is the information content.

**Prediction.** Take each position's final 512 numbers and project to 1024 vocabulary logits. Compare against the actual next tokens. That's the loss.

## Residual Connections

When attention updates a position's 512 numbers, it doesn't replace them — it adds to them. The attention output is a correction: "here's what I learned from looking around, add this to what you already know."

If you replace, a bad layer can destroy good information. If you add, the worst a layer can do is add zeros (contribute nothing). Much more stable to train.

```python
x = x + attn_scale * attn_out
```

The `+` is the residual connection. MLP works the same way.

## Normalization

Before attention and MLP see the 512 numbers, those numbers get normalized — rescaled to a consistent range. Without this, after a few blocks of adding things together, values could drift to wildly different scales across positions. That makes everything downstream unstable.

RMS norm rescales the 512 numbers so they have consistent magnitude.

```python
attn_out = self.attn(self.attn_norm(x))   # normalize, then attend
self.mlp(self.mlp_norm(x))                 # normalize, then MLP
```

Necessary bookkeeping, not conceptually deep.

## The x0 Mix

Every block receives `x` (the current evolving representation) and `x0` (the original embedding, never modified). At the start of each block, the model blends them:

```python
x = mix[0] * x + mix[1] * x0
```

After several blocks, `x` has been heavily processed and might drift too far from the original signal. The mix lets each block peek at the raw input if that helps — "I want 90% of the processed version but let me see 10% of the original too."

Initialized as `mix[0]=1, mix[1]=0` (ignore x0 entirely). The model learns the right ratio per block during training.

## Skip Connections (U-Net Pattern)

The 9 blocks split into two halves. Blocks 0-3 are the "encoder" half, blocks 4-8 are the "decoder" half.

During the encoder half, each block's output gets saved. During the decoder half, those saved outputs get added back in reverse order:

```
Block 4 receives Block 3's saved output
Block 5 receives Block 2's saved output
Block 6 receives Block 1's saved output
Block 7 receives Block 0's saved output
Block 8 gets nothing (ran out)
```

Block 4 is deep in the network, many layers removed from the input. Handing it Block 3's output directly gives it a shortcut back to a shallower representation. Same idea as the x0 mix, but connecting intermediate layers to each other instead of back to the very beginning.

`skip_weights` are learned per-channel scalars that control how much influence each shortcut has.

## MLP

MLP = Multi-Layer Perceptron. Two linear layers with a nonlinearity between them.

A single linear layer can only learn linear relationships (straight decision boundaries). Stacking two linear layers with nothing between them collapses into one linear layer — multiplying two matrices just gives you another matrix. But put a nonlinearity in between and the combination can learn curved, complex relationships.

```python
def forward(self, x: Tensor) -> Tensor:
    x = torch.relu(self.fc(x))      # 512 → 1024, zero out negatives
    return self.proj(x.square())     # square the survivors, 1024 → 512
```

The pattern: expand to a bigger space (512 → 1024) where there's room for more complex transformations, apply the nonlinearity, then compress back down (1024 → 512). The expansion factor is `mlp_mult` in the hyperparameters — the baseline uses 2×, some competitive entries use 3× (1536 hidden).

If attention is "gather information from other tokens," MLP is "process the information at this token." Every position gets the same MLP applied independently — position 0 and position 500 run through the exact same weights. MLP doesn't look at other positions at all. That's attention's job.

In the Illustrated Transformer notes this is the "feed-forward network (position-wise, same weights per position)." MLP is just the common shorthand for the same thing.

## What ReLU Does and What the Numbers Mean

The 512 numbers at each position don't have human-interpretable meaning. They're not "how noun-like is this word" or "how positive is the sentiment." They're numbers the model learned to use internally. Some positive, some negative, their meaning determined entirely by what training found useful.

When the MLP expands from 512 to 1024, that first matrix multiplication (`self.fc(x)`) computes 1024 different "questions" about the 512 input numbers. Each of the 1024 outputs is a weighted sum of the 512 inputs — a different linear combination each time.

Some of those 1024 outputs come out positive, some negative. ReLU zeroes out the negative ones. It acts as a gate: "this question is relevant here, keep it" (positive) or "this question isn't relevant, shut it off" (negative). Different tokens at different positions will have different outputs zeroed out.

A concrete-ish example: maybe one of the 1024 neurons learned weights that fire strongly (positive) when the input represents a verb in past tense, and go negative otherwise. ReLU keeps that signal for "sat" but kills it for "the." The model figures out what detectors are useful through training. Nobody designs them.

The point isn't that ReLU is zeroing out meaningful negatives specifically. It's creating a pattern of on/off activations across 1024 neurons that's different for every input. That pattern of which neurons are active vs silenced is what lets the MLP compute complex nonlinear functions.

Then `.square()` — the surviving positive values get squared. This amplifies large activations relative to small ones. A neuron that fired at 0.9 becomes 0.81, but one that fired at 0.1 becomes 0.01. The strongly activated features dominate the output. This is "ReLU squared" — a common activation function variant.

### What do the 1024 numbers mean?

Nobody designs what those 1024 numbers represent. Before training, the weight matrices are random, so the 1024 numbers are random garbage. During training, backpropagation adjusts the weights to make the final loss go down. Whatever internal representations are useful for predicting the next token, the model develops. Whatever isn't useful fades away.

Same for the 512 embedding numbers. Each dimension doesn't have a label. The model learns that putting similar words near each other in 512-dimensional space and dissimilar words far apart leads to better predictions, so it does that.

The 1024 MLP numbers are even more abstract — a temporary expanded workspace. The model learned that running the 512 numbers through a particular matrix, killing some with ReLU, squaring the survivors, and projecting back to 512 produces better predictions. Why those particular weights work is not something anyone fully understands. This is an open research question called **mechanistic interpretability**.

### Can you see the numbers?

Yes. You can inspect everything at any point. Pause after the embedding and look at all 512 numbers for the word "cat." Pause after Block 3 and see how those 512 numbers changed. Look at the 1024 MLP values and see which ones ReLU zeroed out. Look at the attention weights and see that position 5 is paying 40% attention to position 2 and 10% to position 0.

In PyTorch you can add a `print(x.shape, x)` anywhere in the forward pass and see everything.

Researchers have found individual neurons that activate specifically for Python code, or for text in French, or for numbers. Attention weights are especially interpretable — you can visualize them as a heatmap showing which words attend to which, and the patterns often make intuitive sense (verbs attending to their subjects, pronouns attending to their referents).

But most of the 512 numbers at any given point are not cleanly interpretable. Neuron 347 in Block 6 doesn't correspond to any concept a human would name. It's some learned feature useful for prediction in ways we can't articulate. You can inspect everything while not fully understanding it. That's where the entire AI field is right now.

## Tracing the Data Flow

Say your input is a batch of 8 sequences, each 1024 tokens long.

```python
x = self.tok_emb(input_ids)           # shape: (8, 1024, 512)
x = F.rms_norm(x, (x.size(-1),))
x0 = x                                # save, never changes
```

8 sequences, 1024 positions, 512 numbers per position. `x0` is a frozen snapshot.

GPT calls Block 0:
```python
x = self.blocks[0](x, x0)
```
`x` and `x0` are the same tensor (nothing has happened yet). The `resid_mix` blends them (no-op at init), attention runs, MLP runs. Block 0 returns a new `x`. Still shaped `(8, 1024, 512)`, but now each position's 512 numbers have been updated with information from other positions (via attention) and processed (via MLP).

GPT calls Block 1:
```python
x = self.blocks[1](x, x0)
```
Now `x` is Block 0's output — it's been through one round of attention and MLP. But `x0` is still the original embedding. Block 1's `resid_mix` can blend in raw input signal if that helps. Block 1's attention operates on a richer representation — positions now "know about" their neighbors from Block 0.

This repeats 9 times. The shape never changes, always `(8, 1024, 512)`. What changes is the information content of those 512 numbers at each position. Early blocks capture simple patterns. Later blocks build on those.

Then GPT scores the final `x`:
```python
x = self.final_norm(x).reshape(-1, x.size(-1))  # (8192, 512)
logits_proj = F.linear(x, self.tok_emb.weight)    # (8192, 1024)
```

Each of the 8×1024 = 8192 positions gets projected to 1024 vocabulary logits. "How likely is each of the 1024 possible next tokens?" Cross-entropy compares those predictions against the actual next tokens.

The whole pipeline: raw token IDs → 512-dim embeddings → 9 rounds of "look at context + process" → 1024 vocabulary predictions → compare with reality → loss.

## Block.forward Code

```python
def forward(self, x: Tensor, x0: Tensor) -> Tensor:
    mix = self.resid_mix.to(dtype=x.dtype)
    x = mix[0][None, None, :] * x + mix[1][None, None, :] * x0

    attn_out = self.attn(self.attn_norm(x))
    x = x + self.attn_scale.to(dtype=x.dtype)[None, None, :] * attn_out

    x = x + self.mlp_scale.to(dtype=x.dtype)[None, None, :] * self.mlp(self.mlp_norm(x))
    return x
```

Mix in original → normalize → attend → residual add → normalize → MLP → residual add.

`[None, None, :]` is PyTorch broadcasting — adds two dummy dimensions so the 512-dim vector multiplies across every batch element and sequence position.